# Exp 25: Coarse dropout, downsample 0.4

Card: [exp25_downsample_04.md](../../docs/experiments/exp25_downsample_04.md)

In [ ]:
import matplotlib.pyplot as plt

from core.training.config import ROOT
from core.training.config import VIP
from core.training.cv import CVResult
from core.analysis.gradcam import GradCamFigures
from core.analysis.plots import pr_curves, save_figures, use_style

N_SAMPLE = 16  # exp19 sample
N_POSITIVES = None  # all labeled positives
SEED = 0  # exp19 sample
CONFIG_DIR = ROOT / "configs" / "models"
CONFIG = CONFIG_DIR / "exp25_downsample_04" / "resnet18.yaml"
EXP20 = CONFIG_DIR / "exp20_coarse_dropout_non_vip" / "resnet18.yaml"
EXP16 = CONFIG_DIR / "exp16_downsampling" / "resnet18.yaml"  # exp19 rows drawn from exp16 OOF
FIGURES = ROOT / "docs" / "figures"
PREFIX = "exp25_downsample_04"

# dataviz reference palette, light mode
SERIES = {"exp20": "#2a78d6", "downsample 0.4": "#1baf7a"}
use_style()

## Training

In [ ]:
results = {"exp20": CVResult.load(EXP20)}
results["downsample 0.4"] = CVResult.load(CONFIG)
figures = {}

## Grad-CAM Cat, Dog: exp20 vs downsample 0.4

In [ ]:
cams = {"exp20": GradCamFigures(EXP20), "downsample 0.4": GradCamFigures(CONFIG)}
exp16 = GradCamFigures(EXP16)
for cls in ["Cat", "Dog"]:
    # exp19 rows; smoke OOF holds few rows
    rows = exp16.positive_rows(cls, N_SAMPLE, SEED)
    figures[f"{PREFIX}_grad_cam_{cls.lower()}_compare.png"] = GradCamFigures.compare(cams, cls, rows)
plt.show()

## Grad-CAM VIP: unseen vs trained images

In [ ]:
for cls in VIP:
    for train in (False, True):
        fig = cams["downsample 0.4"].positives(cls, train=train, n=N_POSITIVES)
        split = "train" if train else "validation"
        figures[f"{PREFIX}_grad_cam_{cls.lower()}_{split}.png"] = fig
plt.show()

## Precision-recall curves

In [ ]:
fig = pr_curves(results, colors=SERIES)
figures[f"{PREFIX}_pr_curves.png"] = fig
plt.show()

In [ ]:
save_figures(figures, FIGURES)